#### Task 6: Momentum versus mean-reversion on the same asset

On one asset, implement both a momentum rule (trailing-return sign) and a reversion rule (z-score) over the same sample, and report which one earns the higher net risk-adjusted return.

**Stretch:** build a switch that selects between the two using a trend or volatility filter (for example, run momentum when the 200-day average is rising and reversion when it is flat), and test whether the switched rule beats both fixed rules out of sample. Report the comparison using the risk-adjusted metrics of Lesson 8.

##### Report

Over 2005–2025, momentum had the highest net Sharpe. The switch earned a slightly higher total return than momentum (322.2% versus 319.3%), but its higher annualized volatility (14.2% versus 11.0%) left it with a lower Sharpe.

Over the 2016–2025 test period, the switch met the stretch objective. Its net Sharpe of 0.949 exceeded momentum’s by 0.057 and mean reversion’s by 0.602. The improvement over the stronger fixed rule was modest, and momentum still led over the full sample.


In [ ]:
import numpy as np
import pandas as pd


TICKER = "SPY"
START = "2005-01-01"
END = "2026-01-01"  # Exclusive download date
TEST_START = "2016-01-01"
COST = 0.001  # 10 basis points per entry or exit
INITIAL_CAPITAL = 10_000
TRADING_DAYS = 252


def make_positions(close):
    """Return positions to hold from each day's open to the next open."""
    trailing_return = close.pct_change(60)
    momentum = (trailing_return > 0).astype(int)

    rolling_mean = close.rolling(20).mean()
    rolling_std = close.rolling(20).std()
    z_score = (close - rolling_mean) / rolling_std

    # An entry or exit changes the position. Otherwise carry it forward.
    reversion = pd.Series(np.nan, index=close.index)
    reversion.loc[z_score < -1.5] = 1
    reversion.loc[z_score >= 0] = 0
    reversion = reversion.ffill().fillna(0).astype(int)

    ma_slope = close.rolling(200).mean().pct_change(20)
    switched = momentum.where(ma_slope > 0.005, reversion)

    signals = pd.DataFrame({
        "Momentum": momentum,
        "Mean reversion": reversion,
        "Switch": switched,
    })
    # Common warm-up for all rules. A close signal is tradable the next day.
    return signals.loc[ma_slope.notna()].shift(1).dropna().astype(int)


def backtest(position, open_return):
    """Start in cash and value any remaining position at the final open."""
    trades = position.diff().abs().fillna(position)
    gross = position * open_return
    net = (1 - COST * trades) * (1 + gross) - 1

    equity = INITIAL_CAPITAL * (1 + net).cumprod()
    drawdown = equity / equity.cummax().clip(lower=INITIAL_CAPITAL) - 1
    years = len(net) / TRADING_DAYS
    cagr = (equity.iloc[-1] / INITIAL_CAPITAL) ** (1 / years) - 1
    daily_vol = net.std()
    max_drawdown = drawdown.min()

    # Sharpe assumes a zero risk-free rate.
    return {
        "Total return (%)": 100 * (equity.iloc[-1] / INITIAL_CAPITAL - 1),
        "CAGR (%)": 100 * cagr,
        "Volatility (%)": 100 * daily_vol * np.sqrt(TRADING_DAYS),
        "Sharpe": net.mean() / daily_vol * np.sqrt(TRADING_DAYS) if daily_vol > 0 else np.nan,
        "Max drawdown (%)": 100 * max_drawdown,
        "Trades": int(trades.sum()),
    }


def main():
    import yfinance as yf

    # Extra price history lets all rules start trading at the sample boundary.
    prices = yf.download(
        TICKER, start="2003-01-01", end=END, auto_adjust=True,
        multi_level_index=False, progress=False,
    ).dropna(subset=["Open", "Close"])
    if prices.empty:
        raise RuntimeError("No SPY data downloaded. Check your connection and retry.")

    positions = make_positions(prices["Close"])
    open_return = (prices["Open"].shift(-1) / prices["Open"] - 1).dropna()
    positions = positions.loc[START:]
    positions = positions.loc[positions.index.intersection(open_return.index)]
    test_positions = positions.loc[TEST_START:]

    for label, sample in [("Full sample", positions), ("Out of sample", test_positions)]:
        if sample.empty:
            raise RuntimeError(f"No usable prices for {label.lower()}.")
        table = pd.DataFrame({
            rule: backtest(sample[rule], open_return.loc[sample.index])
            for rule in sample
        }).T
        table["Trades"] = table["Trades"].astype(int)
        print(f"\n{label} ({sample.index[0]:%Y-%m-%d} to {sample.index[-1]:%Y-%m-%d}, interval start dates)")
        print(table.round(3).to_string())

        fixed = table.loc[["Momentum", "Mean reversion"], "Sharpe"]
        if fixed.isna().any():
            print("At least one fixed-rule Sharpe is undefined.")
        elif fixed.nunique() == 1:
            print("The fixed rules tie on net Sharpe.")
        else:
            print(f"Higher fixed-rule net Sharpe: {fixed.idxmax()}")

        if label == "Out of sample" and fixed.notna().all():
            switch_sharpe = table.loc["Switch", "Sharpe"]
            if pd.isna(switch_sharpe):
                print("The switch's net Sharpe is undefined.")
            else:
                print(f"Switch beats both fixed rules on net Sharpe: {switch_sharpe > fixed.max()}")


if __name__ == "__main__":
    main()



Full sample (2005-01-03 to 2025-12-30, interval start dates)
                Total return (%)  CAGR (%)  Volatility (%)  Sharpe  Max drawdown (%)  Trades
Momentum                 319.309     7.078          10.982   0.678           -20.658     205
Mean reversion            77.164     2.766          13.553   0.269           -36.541     240
Switch                   322.151     7.113          14.177   0.556           -40.857     229
Higher fixed-rule net Sharpe: Momentum

Out of sample (2016-01-04 to 2025-12-30, interval start dates)
                Total return (%)  CAGR (%)  Volatility (%)  Sharpe  Max drawdown (%)  Trades
Momentum                 162.103    10.145          11.595   0.892           -20.658      87
Mean reversion            42.316     3.602          12.454   0.347           -30.001     112
Switch                   215.961    12.228          13.056   0.949           -18.014      91
Higher fixed-rule net Sharpe: Momentum
Switch beats both fixed rules on net Sharpe: True
